# **Step 0：挂载 Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Step 1：设路径和基本参数**
这一步在做什么
* set_seed(42)：固定随机性，保证每次结果更一致
* DATA_ROOT / TRAIN_DIR / ...：把路径写清楚
* CKPT_DIR：单独建一个保存 checkpoint 的文件夹
* BATCH_SIZE=32：先用一个比较稳的设置测速度



In [1]:
import os
import time
import random
import numpy as np
import torch

# ========= 1. 固定随机种子（保证可复现） =========
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ========= 2. 路径 =========
DATA_ROOT = "/content/drive/MyDrive/MLP/Project5/dataset_project5"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

# checkpoint 保存路径
CKPT_DIR = "/content/drive/MyDrive/MLP/Project5/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

LATEST_CKPT = os.path.join(CKPT_DIR, "resnet18_stage1_latest.pth")

# ========= 3. 设备 =========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ========= 4. 先设一个安全 batch size =========
BATCH_SIZE = 32
NUM_WORKERS = 2
IMG_SIZE = 224
LR = 1e-3

Device: cuda
GPU: Tesla T4


# **Step 2：读数据并检查**

检查图片是否有损坏

In [8]:
import os
from PIL import Image

DATA_ROOT = "/content/drive/MyDrive/MLP/Project5/dataset_project5"

folders_to_check = [
    os.path.join(DATA_ROOT, "train"),
    os.path.join(DATA_ROOT, "val"),
    os.path.join(DATA_ROOT, "test"),
    os.path.join(DATA_ROOT, "test_jpeg_q30"),
]

bad_files = []

def scan_images(root_dir):
    for current_root, _, files in os.walk(root_dir):
        for fname in files:
            path = os.path.join(current_root, fname)
            try:
                with Image.open(path) as img:
                    img.verify()  # 检查文件是否损坏
                # verify 后需要重新打开一次，确保 convert/load 也没问题
                with Image.open(path) as img:
                    img.convert("RGB")
            except Exception as e:
                bad_files.append((path, str(e)))

for folder in folders_to_check:
    print(f"Scanning: {folder}")
    scan_images(folder)

print(f"\nTotal bad files found: {len(bad_files)}")
for path, err in bad_files[:20]:
    print(path, "->", err)

Scanning: /content/drive/MyDrive/MLP/Project5/dataset_project5/train


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Scanning: /content/drive/MyDrive/MLP/Project5/dataset_project5/val
Scanning: /content/drive/MyDrive/MLP/Project5/dataset_project5/test
Scanning: /content/drive/MyDrive/MLP/Project5/dataset_project5/test_jpeg_q30

Total bad files found: 2
/content/drive/MyDrive/MLP/Project5/dataset_project5/train/real/0038.jpg -> image file is truncated (0 bytes not processed)
/content/drive/MyDrive/MLP/Project5/dataset_project5/val/real/13021.jpg -> image file is truncated (0 bytes not processed)


删除损坏图片

In [9]:
bad_paths = [
    "/content/drive/MyDrive/MLP/Project5/dataset_project5/train/real/0038.jpg",
    "/content/drive/MyDrive/MLP/Project5/dataset_project5/val/real/13021.jpg",
]

import os

for path in bad_paths:
    if os.path.exists(path):
        os.remove(path)
        print(f"Deleted: {path}")
    else:
        print(f"Not found: {path}")

Deleted: /content/drive/MyDrive/MLP/Project5/dataset_project5/train/real/0038.jpg
Deleted: /content/drive/MyDrive/MLP/Project5/dataset_project5/val/real/13021.jpg


In [ ]:
统计剩余图片数量

In [10]:
for split in ["train", "val", "test", "test_jpeg_q30"]:
    for cls in ["real", "fake"]:
        folder = os.path.join(DATA_ROOT, split, cls)
        print(split, cls, len(os.listdir(folder)))

train real 2549
train fake 2550
val real 449
val fake 450
test real 450
test fake 450
test_jpeg_q30 real 450
test_jpeg_q30 fake 450


In [11]:
NUM_WORKERS = 0

In [ ]:
读取图片

In [12]:
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# ========= 5. 数据预处理 =========
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ========= 6. 数据集 =========
train_dataset = ImageFolder(TRAIN_DIR, transform=transform)
val_dataset = ImageFolder(VAL_DIR, transform=transform)
test_dataset = ImageFolder(TEST_DIR, transform=transform)

print("Class mapping:", train_dataset.class_to_idx)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

# ========= 7. DataLoader =========
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ========= 8. 检查一个 batch =========
images, labels = next(iter(train_loader))
print("One batch image shape:", images.shape)
print("One batch label shape:", labels.shape)

Class mapping: {'fake': 0, 'real': 1}
Train size: 5099
Val size: 899
Test size: 900
One batch image shape: torch.Size([32, 3, 224, 224])
One batch label shape: torch.Size([32])


# **Step 3：建 baseline 模型**

In [13]:
import torch.nn as nn
from torchvision import models

# ========= 9. 建 ResNet18 =========
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 先冻结 backbone（第一阶段只为测速和sanity check）
for param in model.parameters():
    param.requires_grad = False

# 改最后分类层为2分类
model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)

# ========= 10. 损失函数和优化器 =========
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=LR)

print(model.fc)

Linear(in_features=512, out_features=2, bias=True)


# **Step 4：Checkpoint 保存 / 恢复**

In [14]:
# ========= 11. checkpoint 保存函数 =========
def save_checkpoint(epoch, model, optimizer, loss, ckpt_path):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss,
    }
    torch.save(checkpoint, ckpt_path)
    print(f"[Checkpoint saved] epoch={epoch} -> {ckpt_path}")

# ========= 12. checkpoint 恢复函数 =========
def load_checkpoint(model, optimizer, ckpt_path, device):
    if os.path.exists(ckpt_path):
        checkpoint = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        last_loss = checkpoint["loss"]
        print(f"[Checkpoint loaded] resume from epoch {start_epoch}")
        return model, optimizer, start_epoch, last_loss
    else:
        print("[No checkpoint found] start from scratch")
        return model, optimizer, 0, None

# **Step 5：跑 1 个 epoch 并计时**

In [15]:
# ========= 13. 如有 checkpoint 则恢复 =========
model, optimizer, start_epoch, last_loss = load_checkpoint(
    model, optimizer, LATEST_CKPT, device
)

# ========= 14. 训练一个 epoch =========
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    total_samples = 0

    start_time = time.time()

    for step, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

        # 每50步打印一次进度
        if (step + 1) % 50 == 0:
            print(f"Step [{step+1}/{len(loader)}] - Loss: {loss.item():.4f}")

    epoch_loss = running_loss / total_samples
    elapsed = time.time() - start_time

    return epoch_loss, elapsed

[No checkpoint found] start from scratch


In [16]:
# ========= 15. 先只跑 1 个 epoch =========
epoch = start_epoch
print(f"\nStart training: epoch {epoch}")

epoch_loss, elapsed = train_one_epoch(
    model, train_loader, criterion, optimizer, device
)

print(f"\nEpoch {epoch} finished")
print(f"Train Loss: {epoch_loss:.4f}")
print(f"Time: {elapsed:.2f} sec ({elapsed/60:.2f} min)")

# 保存 checkpoint
save_checkpoint(epoch, model, optimizer, epoch_loss, LATEST_CKPT)


Start training: epoch 0
Step [50/160] - Loss: 0.4568
Step [100/160] - Loss: 0.5545
Step [150/160] - Loss: 0.3601

Epoch 0 finished
Train Loss: 0.5425
Time: 500.04 sec (8.33 min)
[Checkpoint saved] epoch=0 -> /content/drive/MyDrive/MLP/Project5/checkpoints/resnet18_stage1_latest.pth
